<a href="https://colab.research.google.com/github/shravan1808/ML_SERIES/blob/Main/09_Train_Test_Splitting_Stratified_Sampling_Audit/notebook/Project_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

In [6]:
demographics_data = {
    'Patient_ID': [f'PAT_{500+i}' for i in range(50)],
    'Age': [45, 62, 29, 71, 55, 38, 67, 42, 59, 31,
            48, 65, 26, 73, 52, 35, 69, 41, 58, 33,
            50, 61, 28, 70, 54, 39, 66, 44, 60, 30,
            47, 64, 27, 72, 53, 36, 68, 43, 57, 32,
            49, 63, 25, 74, 51, 37, 70, 40, 56, 34],
    'Gender': ['F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M',
               'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M',
               'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M',
               'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M',
               'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M']
}

In [7]:
vitals_data = {
    'Patient_ID': [f'PAT_{500+i}' for i in range(50)],
    'BMI': [24.5, 31.2, 21.0, 34.8, 28.1, 23.4, 32.5, 26.0, 29.8, 22.1,
            25.3, 30.8, 20.5, 35.2, 27.9, 23.0, 33.1, 25.8, 29.1, 21.8,
            26.1, 31.5, 20.8, 34.2, 28.5, 23.9, 32.0, 26.4, 29.5, 22.5,
            25.0, 31.0, 20.2, 35.6, 27.5, 23.2, 33.5, 25.5, 28.8, 22.0,
            25.8, 31.8, 20.0, 36.0, 28.2, 23.6, 32.8, 26.2, 29.0, 22.3],
    'Blood_Pressure_Systolic': [120, 145, 115, 160, 132, 122, 150, 128, 138, 118,
                                122, 142, 112, 162, 130, 120, 152, 126, 136, 116,
                                124, 144, 114, 158, 134, 121, 148, 127, 137, 117,
                                121, 141, 111, 165, 129, 119, 154, 125, 135, 115,
                                123, 143, 110, 164, 131, 122, 151, 126, 134, 116]
}

In [8]:
diagnostics_data = {
    'Patient_ID': [f'PAT_{500+i}' for i in range(50)],
    'Biomarker_Level': [1.2, 8.5, 0.8, 9.1, 2.1, 1.0, 7.8, 1.5, 3.2, 0.9,
                        1.4, 7.9, 0.7, 9.5, 2.3, 1.1, 8.2, 1.6, 3.0, 0.8,
                        1.3, 8.1, 0.6, 9.0, 2.0, 1.2, 7.5, 1.4, 3.1, 0.9,
                        1.1, 8.3, 0.5, 9.8, 2.2, 1.0, 8.0, 1.5, 2.9, 0.7,
                        1.3, 8.4, 0.4, 9.6, 2.4, 1.1, 7.7, 1.6, 2.8, 0.8],
    'Has_Condition': [0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
                      0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
                      0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
                      0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
                      0, 1, 0, 1, 0, 0, 1, 0, 0, 0] # 15 Positive (30%), 35 Negative (70%)
}

In [9]:
df_demo = pd.DataFrame(demographics_data)
df_vitals = pd.DataFrame(vitals_data)
df_diag = pd.DataFrame(diagnostics_data)

In [10]:
with open('demographics.csv', 'w', newline='') as file:
    df_demo.to_csv(file, index=False)
with open('vitals.csv', 'w', newline='') as file:
    df_vitals.to_csv(file, index=False)
with open('diagnostics.csv', 'w', newline='') as file:
    df_diag.to_csv(file, index=False)

In [11]:
final_df = (
    df_demo
    .merge(df_vitals, on='Patient_ID')
    .merge(df_diag, on='Patient_ID')
)

In [12]:
print(final_df.head().to_string())

  Patient_ID  Age Gender   BMI  Blood_Pressure_Systolic  Biomarker_Level  Has_Condition
0    PAT_500   45      F  24.5                      120              1.2              0
1    PAT_501   62      M  31.2                      145              8.5              1
2    PAT_502   29      F  21.0                      115              0.8              0
3    PAT_503   71      M  34.8                      160              9.1              1
4    PAT_504   55      F  28.1                      132              2.1              0


In [13]:
X = final_df.drop(columns=['Patient_ID', 'Has_Condition'])
y = final_df['Has_Condition']

In [14]:
imbalance_count = y.value_counts()
imbalance_pct = y.value_counts(normalize=True)

In [15]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print("Non-Stratified Split (80/20):")
print(f"- {'Training Set Size':<20} : {len(y_train)} records")
print(f"   - {'Negative (0)':<17} : {(y_train.value_counts().get(0,0))} ({y_train.value_counts(normalize=True).get(0,0.0)*100}%)")
print(f"   - {'Positive (1)':<17} : {(y_train.value_counts().get(1,0))} ({y_train.value_counts(normalize=True).get(1,0.0)*100}%)")
print(f"- {'Testing Set Size':<20}  : {len(y_test)} records")
print(f"   - {'Negative (0)':<17} : {(y_test.value_counts().get(0,0))} ({y_test.value_counts(normalize=True).get(0,0.0)*100}%)")
print(f"   - {'Positive (1)':<17} : {(y_test.value_counts().get(1,0))} ({y_test.value_counts(normalize=True).get(1,0.0)*100}%)")

Non-Stratified Split (80/20):
- Training Set Size    : 40 records
   - Negative (0)      : 27 (67.5%)
   - Positive (1)      : 13 (32.5%)
- Testing Set Size      : 10 records
   - Negative (0)      : 8 (80.0%)
   - Positive (1)      : 2 (20.0%)


In [16]:
X_train_st,X_test_st,y_train_st,y_test_st = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("Stratified Split (80/20):")
print(f"- {'Training Set Size':<20} : {len(y_train_st)} records")
print(f"   - {'Negative (0)':<17} : {(y_train_st.value_counts().get(0,0))} ({y_train_st.value_counts(normalize=True).get(0,0.0)*100}%)")
print(f"   - {'Positive (1)':<17} : {(y_train_st.value_counts().get(1,0))} ({y_train_st.value_counts(normalize=True).get(1,0.0)*100}%)")
print(f"- {'Testing Set Size':<20}  : {len(y_test_st)} records")
print(f"   - {'Negative (0)':<17} : {(y_test_st.value_counts().get(0,0))} ({y_test_st.value_counts(normalize=True).get(0,0.0)*100}%)")
print(f"   - {'Positive (1)':<17} : {(y_test_st.value_counts().get(1,0))} ({y_test_st.value_counts(normalize=True).get(1,0.0)*100}%)")

Stratified Split (80/20):
- Training Set Size    : 40 records
   - Negative (0)      : 28 (70.0%)
   - Positive (1)      : 12 (30.0%)
- Testing Set Size      : 10 records
   - Negative (0)      : 7 (70.0%)
   - Positive (1)      : 3 (30.0%)


In [19]:
X_train_shape = X_train.shape
X_test_shape = X_test.shape
y_train_shape = y_train.shape
y_test_shape = y_test.shape
print("Final Matrix Dimensions:")
print(f"- {'X_train Shape':<15} : {X_train_shape}")
print(f"- {'X_test Shape':<15} : {X_test_shape}")
print(f"- {'y_train Shape':<15} : {y_train_shape}")
print(f"- {'y_test Shape':<15} : {y_test_shape}")

Final Matrix Dimensions:
- X_train Shape   : (40, 5)
- X_test Shape    : (10, 5)
- y_train Shape   : (40,)
- y_test Shape    : (10,)


In [30]:
print(f"{'='*10} TRAIN-TEST SPLITTING & STRATIFIED SAMPLING AUDIT {'='*10}\n")
print(f"{'Master Diagnostic Dataset Records':<34} : {len(final_df)}")
print(f"{'Target Label Class Ratio':<34} : {imbalance_pct.get(0,0.0)*100}% Negative(0) | {imbalance_pct.get(1,0.0)*100}% Positive(1)")
print("\nSampling Methodology Comparison:")
print(f"- {'Standard Random Split(80/20)':<32} : Test set shifted to {y_test.value_counts(normalize=True).get(0,0.0)*100}% Negative / {y_test.value_counts(normalize=True).get(1,0.0)*100}% Positive (Distribution Drift)")
print(f"- {'Stratified Random Split (80/20)':<32} : Test set perfectly maintained {y_test_st.value_counts(normalize=True).get(0,0.0)*100}% Negative / {y_test_st.value_counts(normalize=True).get(1,0.0)*100}% Positive ratio")
print("\nFinal Prepared Matrices:")
print(f"- {'Training Features (X_train)':<32} : {X_train_shape[0]} Rows, {X_train_shape[1]} Numerical/Categorical Features")
print(f"- {'Testing Features (X_test)':<32} : {X_test_shape[0]} Rows, {X_test_shape[1]} Numerical/Categorical Features")
print("""\nConclusion:
For imbalanced datasets, Stratified Sampling is mandatory to eliminate class distribution drift between training and evaluation phases.""")


========== TRAIN-TEST SPLITTING & STRATIFIED SAMPLING AUDIT ==========

Master Diagnostic Dataset Records  : 50
Target Label Class Ratio           : 70.0% Negative(0) | 30.0% Positive(1)

Sampling Methodology Comparison:
- Standard Random Split(80/20)     : Test set shifted to 80.0% Negative / 20.0% Positive (Distribution Drift)
- Stratified Random Split (80/20)  : Test set perfectly maintained 70.0% Negative / 30.0% Positive ratio

Final Prepared Matrices:
- Training Features (X_train)      : 40 Rows, 5 Numerical/Categorical Features
- Testing Features (X_test)        : 10 Rows, 5 Numerical/Categorical Features

Conclusion:
For imbalanced datasets, Stratified Sampling is mandatory to eliminate class distribution drift between training and evaluation phases.
